# KTP Associate Skills Exercise
**Nature Broking Ltd × University of Strathclyde**

---

This exercise assesses your ability to apply NLP and textual analysis to corporate sustainability disclosures in the context of the voluntary carbon market. It is designed to take approximately **2–3 hours**.

You will:
- Characterise a corpus of 10-K filings from 62 US firms across eight sectors — Task 1
- Assess the quality of two real carbon offset projects — Task 2
- Conduct your own investigation into a question of your choice — Task 3

**AI assistance is permitted.** After each task, complete the AI disclosure cell: paste your prompts verbatim, summarise what the AI suggested, and explain what you accepted, modified, or rejected. The panel will use these disclosures at interview.

**Before you do anything else, run Step 0.**

In [ ]:
# ── Step 0: Environment Setup ─────────────────────────────────────────────
# Run this cell at the start of every new Colab session.
# It clones the exercise repository and loads all required libraries.
# Colab resets its environment on disconnect — always run this cell first.

import subprocess, os, sys

REPO_URL = "https://github.com/iamjamesbowden/KTP-Exercise.git"
REPO_DIR = "/content/KTP-Exercise"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repository cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True)
    print("Repository updated.")

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "scikit-learn", "nltk", "wordcloud", "-q"],
    check=True
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import re
import nltk
from collections import Counter
import warnings
warnings.filterwarnings("ignore")

for res in ["punkt", "stopwords", "wordnet", "omw-1.4",
            "averaged_perceptron_tagger"]:
    try:
        nltk.download(res, quiet=True)
    except Exception:
        pass

CORPUS_PATH  = f"{REPO_DIR}/data/corpus.csv"
LM_PATH      = f"{REPO_DIR}/data/lm_dictionary.csv"
HARVARD_PATH = f"{REPO_DIR}/data/harvard_iv_dictionary.csv"

corpus = pd.read_csv(CORPUS_PATH)

print(f"\nCorpus loaded: {len(corpus):,} rows")
print(f"  Firms     : {corpus['firm'].nunique()}")
print(f"  Categories: {sorted(corpus['category'].unique())}")
print(f"  Sections  : {sorted(corpus['section'].unique())}")
print(f"  Years     : {sorted(corpus['year'].unique())}")

APPLICANT_NAME = input("\nEnter your full name: ").strip()
print(f"\nWelcome, {APPLICANT_NAME}. You are ready to begin.")

---

## Optional: LLM Access via OpenRouter

If you have been provided with an OpenRouter API key, you can use it to call an LLM directly within this notebook. This is entirely optional — all tasks can be completed without it.

**Your key is pre-loaded with $5 of credits. Please be economical.** Stick to the recommended models listed in the cell below. Avoid expensive models, which will exhaust the credits quickly and may leave you unable to complete the exercise.

**To set up your key:**
1. Click the **key icon** (🔑) in the left-hand sidebar of Colab to open Secrets
2. Click **Add new secret**
3. Set the name to `OPENROUTER_API_KEY` and paste your key as the value
4. Enable notebook access by toggling the switch next to the secret

**To use the key**, run the cell below. It defines a `call_llm(prompt)` helper you can call anywhere in the notebook. If no key is found, the function will raise a clear error — simply skip it and proceed without LLM access.

In [ ]:
# Optional LLM helper — requires OPENROUTER_API_KEY in Colab Secrets
# Skip this cell if you do not have a key.

import requests as _requests

def call_llm(prompt, model="anthropic/claude-haiku-4.5", max_tokens=1000):
    """
    Call an LLM via OpenRouter. Returns the response as a string.
    Logs the call so you can paste it into the AI disclosure cell.

    Parameters
    ----------
    prompt      : str   — your prompt (paste verbatim into the disclosure cell)
    model       : str   — OpenRouter model ID (default: Claude Haiku 4.5)
    max_tokens  : int   — maximum tokens in the response
    """
    try:
        from google.colab import userdata
        api_key = userdata.get('OPENROUTER_API_KEY')
    except Exception:
        raise RuntimeError(
            "OPENROUTER_API_KEY not found in Colab Secrets. "
            "See the setup instructions in the cell above."
        )

    resp = _requests.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={"Authorization": f"Bearer {api_key}",
                 "Content-Type": "application/json"},
        json={"model": model,
              "messages": [{"role": "user", "content": prompt}],
              "max_tokens": max_tokens},
        timeout=60,
    )
    resp.raise_for_status()
    result = resp.json()["choices"][0]["message"]["content"]

    print("── LLM call logged ──────────────────────────────────────")
    print(f"Model : {model}")
    print(f"Prompt: {prompt[:200]}{'...' if len(prompt)>200 else ''}")
    print(f"Response ({len(result.split())} words):")
    print(result)
    print("─────────────────────────────────────────────────────────")
    return result

print("call_llm() helper ready. Use call_llm(prompt) anywhere in this notebook.")
print("The default model is anthropic/claude-haiku-4.5 — if you are happy with this, you do not need to change anything.")
print("To use a different model, pass model='model-id' as an argument. Available models: https://openrouter.ai/models")
print("Example alternative: openai/gpt-5-mini")

---

## Research Context

### The Voluntary Carbon Market

The voluntary carbon market (VCM) enables organisations to purchase carbon credits to compensate for greenhouse gas emissions that cannot yet be eliminated through operational decarbonisation. Unlike compliance markets — such as the EU Emissions Trading System — participation in the VCM is driven by corporate sustainability commitments: net zero pledges, Science Based Targets (SBTi), and TCFD-aligned disclosure requirements, rather than by regulatory obligation.

In recent years the quality and credibility of VCM activity has come under significant scrutiny. Investigative reporting and academic research have raised concerns about offset projects that deliver substantially fewer emissions reductions than claimed, and about corporations using carbon credits as a substitute for genuine decarbonisation rather than as a last-resort mechanism for residual emissions. This has prompted the emergence of market integrity bodies — notably the Integrity Council for the Voluntary Carbon Market (ICVCM) and the Voluntary Carbon Markets Integrity Initiative (VCMI) — which publish standards for what constitutes a credible, high-quality carbon credit and a credible corporate claim.

### Nature Broking Ltd

Nature Broking is a carbon credit brokerage firm that curates and brokers premium voluntary carbon credits for corporate clients. Rather than offering transactional, volume-driven credit sales, Nature Broking positions itself as a strategic partner: conducting due diligence aligned to ICVCM standards, constructing diversified portfolios of high-quality credits, and advising clients on how to align their offset strategies with credible, science-based approaches. Their clients are corporations pursuing net zero commitments who need confidence that the credits they purchase represent genuine, permanent, and additional emissions reductions — and that their claims will withstand public and regulatory scrutiny.

### Information Asymmetry in the Voluntary Carbon Market

A central economic challenge in the VCM is information asymmetry between buyers and sellers of carbon credits. Buyers — the corporate firms in this corpus — typically cannot directly verify whether a credit represents a genuine, additional, and permanent emissions reduction. This mirrors Akerlof's (1970) "market for lemons" problem: when buyers cannot distinguish high-quality from low-quality products, adverse selection can cause markets to fail to reward quality. In the VCM, concerns about credit quality have depressed prices for high-quality credits and attracted regulatory scrutiny, with some evidence suggesting that buyers unable to distinguish quality tiers default to lower-cost options regardless of environmental integrity. Third-party standards such as the Integrity Council for the Voluntary Carbon Market (ICVCM) and the Social Carbon Standard, and specialist intermediaries such as Nature Broking, can be understood as institutional responses to this information problem — mechanisms designed to reduce asymmetry and allow credible projects to be identified and priced accordingly.

### Research Framing

A central challenge for firms operating in or adjacent to the VCM is assessing the seriousness of corporate climate commitments from public disclosure documents. Corporate filings vary enormously in how specifically and verifiably they discuss their carbon strategies — some provide detailed, quantified, and framework-referenced disclosures; others offer only vague aspirational language. Understanding this variation, and what it signals about genuine commitment versus surface-level compliance, is directly relevant to how Nature Broking identifies and advises its clients.

The question motivating this exercise is: **how specifically and credibly do firms across different sectors disclose their carbon and climate strategies in public filings — and what does the language of these disclosures reveal about the seriousness of their commitments and their readiness to engage credibly with the voluntary carbon market?**

---

## About the Data

All data files are available in the repository's `data/` folder:
**https://github.com/iamjamesbowden/KTP-Exercise/tree/main/data**

Step 0 clones the repository to `/content/KTP-Exercise`, so all files are accessible at `/content/KTP-Exercise/data/` once Step 0 has been run. The paths `CORPUS_PATH`, `LM_PATH`, and `HARVARD_PATH` are set automatically in Step 0.

---

### corpus.csv
The main analysis corpus. **569 rows**, each representing one section of one 10-K filing.

| Column | Description |
|---|---|
| `firm` | Company name |
| `ticker` | Stock ticker |
| `category` | Sector: `oil_gas`, `utility`, `renewable`, `airline`, `big_tech`, `financial`, `consumer`, `industrial` |
| `year` | Filing year (2019–2023) |
| `section` | `item_1a` (Risk Factors) or `item_7` (MD&A) |
| `text` | Full text of the section |
| `word_count` | Word count of the section |

This is your primary dataset for Tasks 1 and 3. It is loaded automatically in Step 0 as `corpus`.

```python
corpus = pd.read_csv(CORPUS_PATH)   # already done in Step 0
```

---

### vcm_buyers.csv
A subset of the corpus containing only the non-energy VCM buyer firms — `airline`, `big_tech`, `financial`, `consumer`, and `industrial` categories. **302 rows**. Same column structure as `corpus.csv`.

Useful if you want to focus your Task 3 analysis on sectors with the most direct VCM purchasing activity, without the energy sector firms.

```python
vcm_buyers = pd.read_csv(f"{REPO_DIR}/data/vcm_buyers.csv")
```

---

### lm_dictionary.csv
The **Loughran-McDonald financial sentiment dictionary** — 2,966 words categorised as Positive, Negative, or Uncertain, designed specifically for financial and corporate disclosure text. More appropriate for this corpus than general-purpose sentiment tools.

Useful for Task 1 vocabulary analysis (does a term carry positive or negative sentiment in a financial context?) or as a starting point for building a scoring approach in Task 3.

```python
lm = pd.read_csv(LM_PATH)   # columns: Word, Positive, Negative, Uncertainty
```

---

### harvard_iv_dictionary.csv
The **Harvard General Inquirer dictionary** — 380 words categorised as Positive or Negative. A general-purpose sentiment lexicon, broader in scope than the LM dictionary but less tailored to financial language.

Useful as a point of comparison against the LM dictionary, or as a baseline for sentiment-based analyses.

```python
hiv = pd.read_csv(HARVARD_PATH)   # columns: Word, Positive, Negative
```

---

## Task 1: Corpus Characterisation

Before analysing any corpus, it is essential to understand its structure, coverage, and vocabulary. This task asks you to produce two specific outputs. The required outputs are prescribed; how you produce them is up to you.

*Suggested time: 35–50 minutes.*

### Output 1.1 — Filing Inventory

Produce a matrix showing filing coverage across firms and years, grouped by category. Your output should make it immediately clear which firm-year combinations are present and whether both sections (Item 1A and Item 7) are available for each.

Include a summary of word count distributions by category and section. A reader should be able to identify any gaps in coverage or data quality considerations at a glance.

In [ ]:
# Output 1.1 — Filing Inventory
# Your code here


### Output 1.2 — Carbon Vocabulary and Disclosure Volume

Analyse how carbon- and climate-related terminology has evolved across the firm categories in the corpus between 2019 and 2023. Produce **two visualisations**:

1. **Term frequency trends** — how the frequency of carbon-related terms changes over time across categories
2. **Disclosure volume** — how the overall volume of climate-relevant text per firm category has changed over time

For the vocabulary analysis, use the seed list below. You may extend it — note any additions in the disclosure cell. Consider flexible matching (plurals, hyphens) and whether to normalise by document length.

**Seed vocabulary**

*Core:* `carbon offset`, `carbon credit`, `carbon neutral`, `carbon neutrality`, `net zero`, `net-zero`, `Scope 1`, `Scope 2`, `Scope 3`, `TCFD`, `SBTi`, `science-based target`, `nature-based solutions`, `sequestration`, `carbon capture`, `greenhouse gas`, `GHG`, `emissions reduction`, `climate target`, `climate commitment`, `carbon footprint`

*VCM-specific:* `Article 6`, `CORSIA`, `REDD+`, `Gold Standard`, `Verra`, `ICVCM`, `VCMI`, `insetting`, `additionality`, `permanence`, `leakage`, `biodiversity credit`, `carbon removal`, `CDR`, `verified carbon`

> **Important:** Several terms in this list — particularly `leakage` and `permanence` — carry different meanings in an energy sector context (pipeline leakage, operational permanence) versus a VCM context (carbon leakage, credit permanence). Treat apparent matches critically and note where context is ambiguous.

> **Note:** Not every VCM-specific term will appear in this corpus. The relative absence of terms like `additionality`, `ICVCM`, or `CORSIA` in corporate 10-K filings is itself a finding worth interpreting.

In the commentary cell below your code, note the most significant patterns (3–4 sentences) and what they suggest about how firms in different sectors are engaging — or not engaging — with VCM-specific language.

In [ ]:
# Output 1.2 — Carbon Vocabulary and Disclosure Volume
# Your code here


In [ ]:
# ── Output 1.2: Commentary (3–4 sentences) ───────────────────────────────

OUTPUT_1_2_COMMENTARY = """
[Your commentary here]
"""

print(OUTPUT_1_2_COMMENTARY)

---

## AI Assistance Disclosure — Task 1

**Did you use an AI assistant for any part of Task 1?** *(double-click to edit — replace `[ ]` with `[x]`)*

- [ ] Yes
- [ ] No

*If yes, complete a block below for each instance.*

**AI use 1**
Tool: *(e.g. ChatGPT, Claude, Copilot)*
Prompt: *(paste verbatim)*
What it suggested: *(brief summary)*
What you did with it: *(accepted / modified / rejected — and why)*

**AI use 2**
Tool:
Prompt:
What it suggested:
What you did with it:

**AI use 3**
Tool:
Prompt:
What it suggested:
What you did with it:

*(Add or remove instances as needed.)*

**Overall reflection** *(2–3 sentences: how did AI assistance shape your approach? What did you decide independently?)*

> *Your reflection here.*

---

## Task 2: Carbon Project Quality Assessment

*Suggested time: 30–40 minutes.*

### Background

In the voluntary carbon market, not all projects are created equal. The quality of a carbon credit depends on how rigorously the underlying project measures, reports, and verifies its emissions reductions — and whether critical quality criteria such as additionality, permanence, and leakage are adequately addressed. For a broker like Nature Broking, whose value proposition rests on due diligence and quality assurance, the ability to critically assess project quality is essential.

Below are brief profiles of two real carbon projects, both in the regenerative agriculture space. Read both carefully before completing Part A.

---

**Project 1: Boomitra Regenerative Agriculture — India**

Developed by Boomitra, a technology-driven carbon project developer, this project supports smallholder farmers across India in adopting regenerative land management practices — including cover cropping, rotational grazing, and reduced tillage — that remove CO₂e from the atmosphere and store it in agricultural soils. It is registered under the Social Carbon Standard, a verification framework developed independently of the Verra or Gold Standard schemes. Measurement, reporting, and verification uses a digital MRV approach combining AI models trained on over one million soil samples with satellite imagery to quantify sequestration at scale. The project is part of a broader portfolio covering 5M+ acres and engaging more than 100,000 farmers across multiple countries, with co-benefits including farmer income, climate resilience, and food security. Credit vintage, CORSIA eligibility, and third-party auditor are not specified in the available summary.

Registry: https://registry.socialcarbon.org/project_details/socialcarbon-9-1695786784824x501444969412689900

---

**Project 2: Regenerate Outcomes Farming Project (ROFP) — United Kingdom**

Developed by Regenerate Outcomes Ltd, a UK-registered company, this grouped project supports farms in Northumberland in transitioning from intensive conventional food production to regenerative systems. It is registered under the Verra Verified Carbon Standard (VCS). The project delivers four services to participating farms: data management and emissions baselining; funding support for capital requirements such as fencing and water infrastructure; mentoring from regenerative agriculture advisors; and marketing of environmental outcomes. Initially comprising 10 farms across upland livestock, lowland mixed, and arable operations, the project generates GHG reductions and removals alongside co-benefits including biodiversity enhancement, water quality improvement, and improved farm economics. Credit vintage, CORSIA eligibility, and third-party auditor are not specified in the available summary.

Registry: https://registry.verra.org/app/projectDetail/VCS/4453

---

### Part A — Project Quality Assessment *(written, 200–300 words)*

In the cell below, compare and contrast the two projects in terms of carbon credit quality. Your response should address:

- What attributes of each project suggest credibility and quality?
- What critical questions remain unanswered, or what aspects of either project are potentially concerning?
- Which project do you consider stronger from a buyer's due diligence perspective, and why?

There is no single correct answer — the panel is looking for evidence that you can distinguish between high and low quality signals in a VCM project, and reason clearly about what matters to a buyer.

In [ ]:
# ── Part A: Your Project Quality Assessment ────────────────────────────
# Replace the placeholder text between the triple quotes with your response.
# Aim for 200–300 words. Do not change anything outside the triple quotes.

PART_A = """
[Your compare and contrast here]
"""

print(PART_A)

---

## AI Assistance Disclosure — Task 2

**Did you use an AI assistant for any part of Task 2?** *(double-click to edit — replace `[ ]` with `[x]`)*

- [ ] Yes
- [ ] No

*If yes, complete a block below for each instance.*

**AI use 1**
Tool: *(e.g. ChatGPT, Claude, Copilot)*
Prompt: *(paste verbatim)*
What it suggested: *(brief summary)*
What you did with it: *(accepted / modified / rejected — and why)*

**AI use 2**
Tool:
Prompt:
What it suggested:
What you did with it:

**AI use 3**
Tool:
Prompt:
What it suggested:
What you did with it:

*(Add or remove instances as needed.)*

**Overall reflection** *(2–3 sentences: how did AI assistance shape your approach? What did you decide independently?)*

> *Your reflection here.*

---

## Task 3: Open Investigation

*Suggested time: 45–60 minutes.*

### Instructions

Design and conduct your own investigation using the corpus loaded in Step 0. This task is an opportunity to demonstrate the analytical questions you find interesting and the methods you are most confident applying.

**Guardrails:**
- You must use the `corpus` dataframe loaded in Step 0
- Your research question must connect specifically to how firms disclose or signal their carbon strategies, how sectors differ in their VCM engagement, or what textual signals are associated with credible versus superficial climate commitments
- You must produce at least one quantitative output — a table, chart, or computed statistic
- You must provide a 2–3 sentence annotation of your key finding directly below your output

Strong responses will show original thinking, clear methodological reasoning, and an ability to situate findings within the VCM context introduced in the briefing. You are not expected to produce a complete research paper — focus on demonstrating how you approach and think through an analytical problem.

In [ ]:
# ── Task 3: Research Question ────────────────────────────────────────────

RESEARCH_QUESTION = """
[State your research question here]
"""

RELEVANCE = """
[2–3 sentences: why this question is relevant to Nature Broking's business]
"""

print("Research question:", RESEARCH_QUESTION)
print("Relevance:", RELEVANCE)

In [ ]:
# Task 3 — Your code here


In [ ]:
# Task 3 — continued


In [ ]:
# Task 3 — continued


In [ ]:
# ── Task 3: Key Finding ──────────────────────────────────────────────────
# In 2–3 sentences, summarise what your output shows.

KEY_FINDING = """
[Your annotation here]
"""

print(KEY_FINDING)

---

## AI Assistance Disclosure — Task 3

**Did you use an AI assistant for any part of Task 3?** *(double-click to edit — replace `[ ]` with `[x]`)*

- [ ] Yes
- [ ] No

*If yes, complete a block below for each instance.*

**AI use 1**
Tool: *(e.g. ChatGPT, Claude, Copilot)*
Prompt: *(paste verbatim)*
What it suggested: *(brief summary)*
What you did with it: *(accepted / modified / rejected — and why)*

**AI use 2**
Tool:
Prompt:
What it suggested:
What you did with it:

**AI use 3**
Tool:
Prompt:
What it suggested:
What you did with it:

*(Add or remove instances as needed.)*

**Overall reflection** *(2–3 sentences: how did AI assistance shape your approach? What did you decide independently?)*

> *Your reflection here.*